# `JEV-Set-150M` — Google Colab Pro Fast Pilot & Validation
**Target Benchmark:** [Jev Decision Index `v0.4.1`](https://huggingface.co/spaces/multimodalart/jev-decision-index)

This pilot notebook executes a focused **end-to-end verification and pilot run** on Colab Pro before committing hours to full-scale training:
1. **GPU & Environment Setup:** Detects A100 / L4 / T4, mounts Google Drive (`/content/drive/MyDrive/jev-set-150m`), and installs dependencies.
2. **Real Weights Verification:** Downloads `cross-encoder/ettin-reranker-150m-v1` and strictly verifies parameter count (`149,014,272`), `4_Dense` linear scorer loading, equivalence vs `transformers.ModernBertModel`, and real tokenizer inference.
3. **Zero-Synthetic Pilot Data Build:** Pulls real samples across dataset groups with `--no-synthetic-fallback` and verifies zero synthetic rows.
4. **Real Teacher Logit Extraction:** Runs teacher inference with automatic checkpoint resumption.
5. **Measured Pilot Ablation:** Executes measured Exp A (backbone gate) & Exp B (topology speedup & permutation equivariance) with paired bootstrap.
6. **Stage 1 Pilot Epoch:** Runs 1 short epoch with real weights and real data.

In [ ]:
# 1. GPU Detection & Google Drive Mount
import os
import shutil
import torch

print("=== CUDA / GPU Status ===")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"Active GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No CUDA GPU detected. Switch runtime via Runtime -> Change runtime type -> GPU.")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/jev-set-150m'
    os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_ROOT}/data/processed', exist_ok=True)
    os.makedirs('data', exist_ok=True)
    if not os.path.islink('checkpoints'):
        if os.path.exists('checkpoints'):
            shutil.rmtree('checkpoints')
        os.symlink(f'{DRIVE_ROOT}/checkpoints', 'checkpoints')
    if not os.path.islink('data/processed'):
        if os.path.exists('data/processed'):
            shutil.rmtree('data/processed')
        os.symlink(f'{DRIVE_ROOT}/data/processed', 'data/processed')
    os.environ['DRIVE_ROOT'] = DRIVE_ROOT
    print(f"Google Drive persistent storage linked: {DRIVE_ROOT}")
except ImportError:
    print("Running outside Google Colab; using local directory.")

In [ ]:
# 2. Install Pinned Dependencies & Decision Index Evaluator
!pip install -q torch>=2.5.1 transformers>=4.48.0 datasets>=3.2.0 accelerate>=1.2.1 safetensors>=0.5.0 scipy>=1.14.1 scikit-learn>=1.6.0 datasketch>=1.6.5 pyyaml>=6.0.2 tqdm>=4.67.1 pytest>=8.0.0
!if [ ! -d external/decision-index ]; then git clone https://github.com/apolinario/decision-index.git external/decision-index && cd external/decision-index && git checkout 25c6ec7b6b5f44a216c63c3a700de3a9c6d6699d && pip install -e . && cd ../..; fi

In [ ]:
# 3. Verify Real Pretrained Weights on Hub & Run Unit Test Suite
!python scripts/verify_real_weights.py
!python scripts/verify_params.py
!pytest tests/ -q

In [ ]:
# 4. Run the Pilot Pipeline (Zero-Synthetic Real Data + Measured Exp A/B + 1-Epoch Pilot)
# You can also run individual steps via bash scripts/run_colab_pro.sh pilot
!bash scripts/run_colab_pro.sh pilot

In [ ]:
# 5. Display Data Summary & Pilot Ablation Results
import json
from pathlib import Path

summary_path = Path("data/processed/data_build_summary.json")
if summary_path.exists():
    print("=== Data Curation Summary ===")
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

abl_path = Path("data/processed/pilot_ablation_results.json")
if abl_path.exists():
    print("=== Pilot Ablation Results ===")
    print(json.dumps(json.loads(abl_path.read_text()), indent=2))